# Phase 25 — RA-MedFiLM component development (25A–25G)

Reconstructed notebook for `phase_25_ra_medfilm_dev`, the umbrella directory for the four BARR components. Each component was trained **separately**, not jointly with the backbone.

> **Status:** reconstructed analysis notebook. It reads persisted artifacts only. No training, no new inference, no protected TEST access.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, r"D:\Softfors\jup\full_doc_report\Scripts")
import remedfilm_guard as GUARD
GUARD.install()

REPO = Path(r"D:\Softfors\jup")
REPORT = REPO / "full_doc_report"
TABLES = REPORT / "tables"
FIG = REPORT / "figures"

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

def fig(subdir, name, width=760):
    """Display a generated figure, failing loudly if it is missing."""
    p = FIG / subdir / f"{name}.png"
    if not p.exists():
        raise FileNotFoundError(f"missing figure: {p}")
    display(Image(filename=str(p), width=width))

def table(path, **kw):
    return pd.read_csv(path, **kw)

GUARD.require()

True

In [2]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified


In [3]:
inv = pd.read_csv(TABLES / "REMEDFILM_COMPONENT_PARAMETERS.csv")
print("component inventory (corrected parameter column)")
inv[["symbol", "model_class", "parameters", "conditional", "role"]].to_string(index=False)
print()
print("sum of components:", int(inv[inv.symbol != "FULL"].parameters.sum()))

component inventory (corrected parameter column)

sum of components: 4837420


In [4]:
# 25A — regional refiner: per-fold selected epochs and final refit.
sel = pd.read_csv(REPO / "reports/tables/phase25a_refiner_selected_epochs.csv")
print("refiner selection records:", len(sel))
sel

refiner selection records: 3


,run,outer_fold,best_epoch,best_early_stop_loss,epochs_completed,stop_reason,resume_count,checkpoint_best_sha256,selection_rule
0,fold0,0,34,0.096024,50,early_stopping_patience_15,0,38803648ba5894c62b8f5fd5675d6e5ab13e416003e82e...,minimum EARLY_STOP total loss
1,fold1,1,27,0.090209,43,early_stopping_patience_15,0,b5f50b743b89b6153ad782c632edd89a7e13285e66573c...,minimum EARLY_STOP total loss
2,fold2,2,18,0.099504,34,early_stopping_patience_15,0,7c0924ff9e53f6403552a76c64425e45850073ff629832...,minimum EARLY_STOP total loss


In [5]:
# 25A — genuine per-epoch refiner history (fold 0 and final refit).
fig_, axes = plt.subplots(1, 2, figsize=(7.2, 2.6))
for ax, (label, path) in zip(axes, [
        ("fold0", REPO / "experiments/phase25a_barr_regional_refiner_v1_1/fold0/history.csv"),
        ("final refit", REPO / "experiments/phase25a_barr_regional_refiner_v1_1/final_refit/history.csv")]):
    h = pd.read_csv(path).sort_values("epoch")
    ax.plot(h.epoch, h.train_loss, color="#0072B2", linewidth=1.2, label="train")
    if "early_stop_dice" in h.columns:
        ax2 = ax.twinx()
        ax2.plot(h.epoch, h.early_stop_dice, color="#D55E00", linewidth=1.1, linestyle="--")
        ax2.set_ylabel("early-stop Dice", color="#D55E00")
        ax2.grid(False)
    ax.set_title(f"R_psi {label}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Train loss")
plt.tight_layout(); plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_38600\1767857067.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [6]:
# 25D — benefit router: 5-fold CV selection.
cv = pd.read_csv(REPO / "reports/tables/phase25d_cv_selected_epochs.csv")
print("router CV folds:", len(cv))
cv

router CV folds: 5


,fold,best_epoch,best_es_huber,total_epochs,termination
0,0,37,0.000056,57,early_stop
1,1,34,0.000038,54,early_stop
2,2,139,0.000056,159,early_stop
3,3,199,0.000046,200,max_epochs
4,4,80,0.000074,100,early_stop


In [7]:
# 25F — post-safety gate: per-fold epochs and acceptance behaviour.
f = pd.read_csv(REPO / "reports/tables/phase25f_final_epoch_rule.csv")
f

,item,value
0,best_epoch_fold0,16
1,best_epoch_fold1,7
2,best_epoch_fold2,66
3,best_epoch_fold3,35
4,best_epoch_fold4,23
5,sorted_best_epochs,7 16 23 35 66
6,median_index,2
7,final_safety_epochs,23
8,final_seed,42


In [8]:
print("ASSEMBLY NOTE")
print("------------")
print("The four components were fitted in sequence, each against a frozen upstream:")
print("  G  (Phase 21)  -> frozen backbone")
print("  R  (Phase 25A) -> fitted on G's outputs, cross-fitted folds")
print("  B  (Phase 25D) -> fitted on G+R OOF gain labels")
print("  S  (Phase 25F) -> fitted on routed candidates")
print()
print("ReMedFiLM FULL is therefore an ASSEMBLED system, never a jointly")
print("end-to-end trained network. No single optimiser ever saw all four modules.")

ASSEMBLY NOTE
------------
The four components were fitted in sequence, each against a frozen upstream:
  G  (Phase 21)  -> frozen backbone
  R  (Phase 25A) -> fitted on G's outputs, cross-fitted folds
  B  (Phase 25D) -> fitted on G+R OOF gain labels
  S  (Phase 25F) -> fitted on routed candidates

ReMedFiLM FULL is therefore an ASSEMBLED system, never a jointly
end-to-end trained network. No single optimiser ever saw all four modules.


## Provenance

- `reports/tables/phase26_component_inventory.csv`
- `experiments/phase25a_*/`, `experiments/phase25d_*/`, `experiments/phase25f_*/`
- `reports/tables/phase25*_selected_epochs.csv`

In [9]:
# Negative control — enforcement must be real, not a flag.
for name, verdict in GUARD.self_test():
    print(f"  {name:28s}: {verdict}")
assert all(v.startswith("blocked") for _, v in GUARD.self_test()), "GUARD FAILED"
print("analysis-only guard verified")

  optimiser construction      : blocked
  AdamW construction          : blocked
  TEST manifest read          : blocked
  write outside report dir    : blocked
analysis-only guard verified
